In [1]:
import pandas as pd
import numpy as np
import joblib

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
import pandas as pd

df = pd.read_csv("fresh_produce_wastage_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (3000, 34)


,batch_id,batch_date,shift,product_type,product_category,supplier_id,quantity_produced_kg,storage_temperature_celsius,humidity_percent,co2_concentration_ppm,...,extreme_weather_event,order_fulfillment_status,quality_score_percent,composite_risk_score,actual_wastage_kg,wastage_percent,wastage_cost_inr,spoilage_class,wastage_risk_category,recommended_corrective_action
0,BATCH-00001,2023-09-19,Morning,Lemon,Citrus,SUP-006,1225.22,15.51,71.23,445.6,...,0,Fulfilled,90.08,2,129.02,10.53,3870.60,Good,Medium,Reduce Storage Temperature
1,BATCH-00002,2020-09-25,Morning,Mango,Fruits,SUP-011,1431.78,13.01,77.96,986.2,...,0,Fulfilled,NaN,1,207.04,14.46,12422.40,Good,Medium,Restore Cold Chain Immediately
2,BATCH-00003,2021-09-29,Night,Leafy Greens,Vegetables,SUP-018,469.12,3.81,60.50,1089.1,...,0,Fulfilled,NaN,1,30.59,6.52,458.85,Good,Low,Quality Re-inspection Required
3,BATCH-00004,2021-07-26,Morning,Pineapple,Fruits,SUP-004,949.68,9.14,74.89,1476.4,...,0,Cancelled,88.68,1,148.25,15.61,6671.25,Bad,Medium,Restore Cold Chain Immediately
4,BATCH-00005,2021-08-14,Morning,Lemon,Citrus,SUP-003,1676.66,11.13,85.47,1458.5,...,1,Delayed,62.94,2,217.80,12.99,6534.00,Good,Medium,Prioritize Dispatch


In [3]:
features = [
    "quantity_produced_kg",
    "storage_temperature_celsius",
    "humidity_percent",
    "co2_concentration_ppm",
    "light_exposure_lux",
    "storage_duration_hours",
    "processing_duration_hours",
    "processing_delay_hours",
    "demand_forecast_kg",
    "inventory_level_kg",
    "delivery_time_days",
    "transport_cost_per_ton_inr",
    "supplier_reliability_score",
    "quality_score_percent"
]

target = "wastage_risk_category"

X = df[features].copy()
y = df[target].copy()

print("Features selected:", len(features))
print("Target:", target)
print("Input shape:", X.shape)
print("Target shape:", y.shape)

Features selected: 14
Target: wastage_risk_category
Input shape: (3000, 14)
Target shape: (3000,)


In [4]:
X = X.fillna(X.median(numeric_only=True))

print("Missing values after preprocessing:")
print(X.isnull().sum().sum())

Missing values after preprocessing:
0


In [5]:
print("Target class distribution:")
print(y.value_counts())

Target class distribution:
wastage_risk_category
Medium    1676
Low       1001
High       323
Name: count, dtype: int64


In [6]:
class_names = sorted(y.unique())

label_to_number = {
    label: index for index, label in enumerate(class_names)
}

number_to_label = {
    index: label for label, index in label_to_number.items()
}

y_numeric = y.map(label_to_number).to_numpy()

print("Class mapping:")
print(label_to_number)

Class mapping:
{'High': 0, 'Low': 1, 'Medium': 2}


In [7]:
np.random.seed(42)

train_indices = []
test_indices = []

for class_number in np.unique(y_numeric):
    class_indices = np.where(y_numeric == class_number)[0]
    np.random.shuffle(class_indices)

    split_point = int(len(class_indices) * 0.8)

    train_indices.extend(class_indices[:split_point])
    test_indices.extend(class_indices[split_point:])

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)

np.random.shuffle(train_indices)
np.random.shuffle(test_indices)

X_train = X.iloc[train_indices].to_numpy(dtype=float)
X_test = X.iloc[test_indices].to_numpy(dtype=float)

y_train = y_numeric[train_indices]
y_test = y_numeric[test_indices]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (2398, 14)
Testing data: (602, 14)


In [8]:
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

train_std[train_std == 0] = 1

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print("Feature scaling completed!")

Feature scaling completed!


In [12]:
class SimpleKNNClassifier:
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        predictions = []
        
        for sample in X:
            distances = np.sqrt(
                np.sum((self.X_train - sample) ** 2, axis=1)
            )
            
            nearest_indices = np.argsort(distances)[:self.k]
            nearest_labels = self.y_train[nearest_indices]
            
            values, counts = np.unique(
                nearest_labels,
                return_counts=True
            )
            
            prediction = values[np.argmax(counts)]
            predictions.append(prediction)
        
        return np.array(predictions)

In [11]:
model = SimpleKNNClassifier(k=5)

model.fit(X_train_scaled, y_train)

print("Model training completed!")

Model training completed!


In [13]:
y_pred = model.predict(X_test_scaled)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

Predictions generated successfully!
Number of predictions: 602


In [14]:
print("Classification Results")
print("=" * 50)

for class_number in sorted(np.unique(y_test)):
    
    true_positive = np.sum(
        (y_test == class_number) & (y_pred == class_number)
    )
    
    actual_positive = np.sum(y_test == class_number)
    predicted_positive = np.sum(y_pred == class_number)
    
    precision = (
        true_positive / predicted_positive
        if predicted_positive > 0 else 0
    )
    
    recall = (
        true_positive / actual_positive
        if actual_positive > 0 else 0
    )
    
    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    
    print("Class:", number_to_label[class_number])
    print("Precision:", round(precision, 3))
    print("Recall:", round(recall, 3))
    print("F1 Score:", round(f1_score, 3))
    print()

Classification Results
Class: High
Precision: 0.194
Recall: 0.108
F1 Score: 0.139

Class: Low
Precision: 0.453
Recall: 0.453
F1 Score: 0.453

Class: Medium
Precision: 0.589
Recall: 0.64
F1 Score: 0.613



In [15]:
model_package = {
    "model": model,
    "train_mean": train_mean,
    "train_std": train_std,
    "label_to_number": label_to_number,
    "number_to_label": number_to_label,
    "features": features
}

print("Model package created successfully!")

Model package created successfully!


In [19]:
class SimpleKNNClassifier:
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        predictions = []
        
        for sample in X:
            distances = np.sqrt(
                np.sum((self.X_train - sample) ** 2, axis=1)
            )
            
            nearest_indices = np.argsort(distances)[:self.k]
            nearest_labels = self.y_train[nearest_indices]
            
            values, counts = np.unique(
                nearest_labels,
                return_counts=True
            )
            
            prediction = values[np.argmax(counts)]
            predictions.append(prediction)
        
        return np.array(predictions)

In [20]:
model = SimpleKNNClassifier(k=5)

model.fit(X_train_scaled, y_train)

print("Model training completed!")

Model training completed!


In [21]:
model_package = {
    "model": model,
    "train_mean": train_mean,
    "train_std": train_std,
    "label_to_number": label_to_number,
    "number_to_label": number_to_label,
    "features": features
}

print("Model package created successfully!")

Model package created successfully!


In [23]:
joblib.dump(
    model_package,
    "wastage_risk_model.joblib"
)

print("Model saved successfully!")
print("File: wastage_risk_model.joblib")

Model saved successfully!
File: wastage_risk_model.joblib


In [24]:
loaded_package = joblib.load("wastage_risk_model.joblib")

print("Saved model loaded successfully!")
print("Stored features:", len(loaded_package["features"]))

Saved model loaded successfully!
Stored features: 14


In [25]:
sample = df[features].iloc[[0]].copy()

sample = sample.fillna(
    df[features].median(numeric_only=True)
)

sample_values = sample.to_numpy(dtype=float)

sample_scaled = (
    sample_values - loaded_package["train_mean"]
) / loaded_package["train_std"]

sample_prediction = loaded_package["model"].predict(sample_scaled)

predicted_class = loaded_package["number_to_label"][
    sample_prediction[0]
]

print("Sample prediction:")
print("Predicted Wastage Risk Category:", predicted_class)

Sample prediction:
Predicted Wastage Risk Category: Medium
